# 12 · Thermodynamic masks: buoyant vs forced updrafts, and parcel (h vs h*) columns

Notebook 11 sorts columns by vertical velocity only. Here the masks also use **thermodynamics**:

* **`buoyant`**: updrafts (W > X at level k) split into **buoyant** (B > 0) and **forced** (B ≤ 0). B is SAM's own buoyancy (van der Drift & O'Gorman eq. 7) relative to the snapshot's horizontal mean, interpolated to the W level. Forced updrafts are, for example, air pushed up at cold-pool edges.
* **`buoyancy_only`**: columns by buoyancy alone at level k (positive / negative / neutral, threshold B0), no W.
* **`parcel_h`**: columns by what their near-surface air *could* do. Lift it keeping its MSE h and compare with the environment's h*(z): positive area = **CAPE proxy**, negative area between LCL and LFC = **CIN proxy**. Groups: `free`, `capped`, `stable`.
* **`parcel_and_w`**: updrafts in `free` columns vs all other updrafts.

Caveats:
* The CAPE/CIN proxies use the standard MSE approximation, b ≈ g (h_p − h*_env)/(c_p T_env). They ignore virtual and condensate-loading effects, and are only counted above each column's LCL (Espy estimate). Use them to rank columns, not as exact CAPE.
* Emanuel's N² / N_m² (section D) stay as diagnostics of the composite profiles. They are not good masks (see the discussion in the session notes).

Workflow: **setup → CONFIG → helpers → MASK BOX → preview → compute → plots**. Edit the helpers or masks freely; the cache notices.

In [ ]:
# RUN THIS CELL FIRST: lets this notebook find the samheat code in this repo (no install needed)
import os, sys, pathlib
def _find_repo():
    starts = [pathlib.Path(os.environ.get("JPY_SESSION_NAME", ".")).resolve(), pathlib.Path.cwd().resolve()]
    for s in starts:
        for d in [s, *s.parents]:
            if (d / "samheat" / "__init__.py").exists():
                return d
    for d in [pathlib.Path("/scratch/ttg2015/Torch/ExtremeHeat/SAM_HEAT"), pathlib.Path.home() / "SAM_HEAT"]:
        if (d / "samheat" / "__init__.py").exists():
            return d
    raise FileNotFoundError("Could not find the SAM_HEAT folder. Set REPO by hand, e.g. REPO = pathlib.Path('/scratch/.../SAM_HEAT')")
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import samheat
print("using samheat from", pathlib.Path(samheat.__file__).parent)

In [ ]:
# ---- CONFIG ----
from samheat.paths import SITE
EXPERIMENT = SITE.work_root / "vdd2025_vddgrid"     # stock-grid runs: "vdd2025_repro"
RV_LIST = [0, 200, 500, 1000, 2000]                 # one stage2_rv<rv> run each
SOURCE = "auto"           # "auto": converted .nc files when every snapshot has one, else .bin3D;
                          # "nc" / "bin" to force one (identical data, so the cache is shared)
TMIN, TMAX = 30, 60       # days of Stage 2 to use (3-hourly 3D snapshots)
INCLUDE_ALL = True        # also composite the whole domain as group "all" (for anomalies)
ZMAX_PLOT = 16000         # m, top of the profile plots (None = model top)
RV_COLORS = ["#1a5292", "#5fa5dd", "#446400", "#d9863b", "#783c25"]   # same as 02_Repro

## Thermodynamic helpers (buoyancy, LCL, CAPE/CIN proxies)

In [ ]:
# ===================== THERMODYNAMIC HELPERS (edit freely; the cache follows this cell's code) =====================
# Every function gets f = dict of 3D arrays (z, y, x) for ONE snapshot, SAM units (T K, q g/kg),
# including the derived MSE and MSE_SAT (kJ/kg), and z = (z,) heights in m.
import numpy as np

G, CP, EPSV = 9.81, 1004., 0.61


def buoyancy(f, include_qp=True):
    """SAM's buoyancy (van der Drift & O'Gorman eq. 7), m/s^2, on scalar levels (z, y, x):
    B = g [ (T - T')/T' (1 + eps_v qv' - qn' - qp') + eps_v (qv - qv') - (qn + qp - qn' - qp') ],
    primes = horizontal mean of this snapshot (the environment). include_qp=False drops rain/snow
    loading, as van der Drift did when relating buoyancy to updrafts."""
    T, qv, qn = f["TABS"], f["QV"] / 1e3, f["QN"] / 1e3
    qp = f["QP"] / 1e3 if include_qp else np.zeros_like(T)
    m = lambda a: a.mean(axis=(1, 2), keepdims=True)
    Tm, qvm, qnm, qpm = m(T), m(qv), m(qn), m(qp)
    return G * ((T - Tm) / Tm * (1 + EPSV * qvm - qnm - qpm) + EPSV * (qv - qvm) - (qn + qp - qnm - qpm))


def buoyancy_at_w(B, k):
    """Buoyancy at the W level k: SAM stores W(k) on the interface below scalar level k."""
    return B[0] if k == 0 else 0.5 * (B[k - 1] + B[k])


def lcl_height(f, z, k_src=0, p_src=1000.):
    """Rough lifting condensation level [m] of the air at level k_src (Espy: 125 m per K of
    dew-point depression; dew point from Bolton's formula at pressure p_src hPa; a 10 hPa error
    in p_src moves the LCL by only ~20 m)."""
    T, qv = f["TABS"][k_src], f["QV"][k_src] / 1e3
    p = p_src
    e = np.maximum(p * qv / (0.622 + qv), 1e-3)                       # hPa
    Td = 243.5 * np.log(e / 6.112) / (17.67 - np.log(e / 6.112)) + 273.15
    return z[k_src] + 125. * np.maximum(T - Td, 0.)


def parcel_cape(f, z, k_src=0, zmax=16000.):
    """CAPE and CIN proxies [J/kg] per column from moist static energy: lift the column's air at
    level k_src keeping its h (MSE), and compare with the snapshot-mean environment h*(z):
        b(z) = g (h_parcel - h*_env(z)) / (c_p T_env(z))      (standard MSE approximation)
    integrated only ABOVE the column's LCL (below it the parcel is unsaturated and this
    approximation does not hold; a well-mixed sub-cloud layer is roughly neutral).
    CAPE = positive area up to zmax; CIN = negative area between the LCL and the first positive level."""
    hp = f["MSE"][k_src]                                                # (y, x) kJ/kg
    hs = f["MSE_SAT"].mean(axis=(1, 2))                                 # (z,) environment h*
    Te = f["TABS"].mean(axis=(1, 2))
    b = G * 1000. * (hp[None] - hs[:, None, None]) / (CP * Te[:, None, None])     # (z, y, x) m/s^2
    dz = np.gradient(z)[:, None, None]
    zz = z[:, None, None]
    above = (zz > lcl_height(f, z, k_src)[None]) & (zz <= zmax)
    pos = (b > 0) & above
    cape = np.where(pos, b * dz, 0.).sum(axis=0)
    zlfc = np.where(pos, zz, np.inf).min(axis=0)                        # first positive level (inf: none)
    cin = np.where(above & (b < 0) & (zz < zlfc[None]), b * dz, 0.).sum(axis=0)
    return cape, cin

## Mask box

In [ ]:
# ===================== MASK BOX =====================
# mask(f, z, **params) -> {group name: boolean (y, x)}; groups must be the same every snapshot.
import inspect
import numpy as np


def buoyant(f, z, k=1, X=0.3, include_qp=True):
    """Updrafts at W level k split by their buoyancy there: driven by it (B > 0) or forced (B <= 0)."""
    w = f["W"][k]
    b = buoyancy_at_w(buoyancy(f, include_qp), k)
    return dict(buoyant_updraft=(w > X) & (b > 0), forced_updraft=(w > X) & (b <= 0),
                downdraft=w < -X, intermediate=np.abs(w) <= X)


def parcel_h(f, z, k_src=0, cape_min=200., cin_max=20., zmax=16000.):
    """Columns by what their near-surface air could do: 'free' (CAPE >= cape_min and CIN >= -cin_max),
    'capped' (CAPE >= cape_min but CIN < -cin_max) or 'stable' (CAPE < cape_min)."""
    cape, cin = parcel_cape(f, z, k_src, zmax)
    ok = cape >= cape_min
    return dict(free=ok & (cin >= -cin_max), capped=ok & (cin < -cin_max), stable=~ok)


def parcel_and_w(f, z, k=1, X=0.3, k_src=0, cape_min=200., cin_max=20., zmax=16000.):
    """Both together: updrafts (W > X at level k) in 'free' columns vs everywhere else."""
    w = f["W"][k]
    cape, cin = parcel_cape(f, z, k_src, zmax)
    free = (cape >= cape_min) & (cin >= -cin_max)
    return dict(free_updraft=(w > X) & free, other_updraft=(w > X) & ~free,
                downdraft=w < -X, intermediate=np.abs(w) <= X)


def buoyancy_only(f, z, k=1, B0=0.01, include_qp=True):
    """Columns by their buoyancy at W level k alone (no W): positive (B > B0), negative (B < -B0),
    neutral (|B| <= B0); B0 in m/s^2 (0.01 m/s^2 = 1 cm/s^2)."""
    b = buoyancy_at_w(buoyancy(f, include_qp), k)
    return dict(positive=b > B0, negative=b < -B0, neutral=np.abs(b) <= B0)


MASKS = dict(buoyant=buoyant, buoyancy_only=buoyancy_only, parcel_h=parcel_h, parcel_and_w=parcel_and_w)

# ---- choose one ----
MASK = "buoyant"
MASK_PARAMS = dict(k=1, X=0.3, include_qp=True)

# the helpers' code is part of the cache key, so editing them recomputes
def _code_id(fn):
    try:
        return inspect.getsource(fn)
    except (OSError, TypeError):                       # source unavailable: fall back to the bytecode
        return repr((fn.__code__.co_code, fn.__code__.co_consts))
KEY_EXTRA = "".join(_code_id(fn) for fn in (buoyancy, buoyancy_at_w, lcl_height, parcel_cape))
print(f"mask: {MASK}{MASK_PARAMS}")

## Preview (cheap: tune thresholds here)

In [ ]:
# ---- Preview: what the thermodynamic fields and the chosen mask look like (one run, a few snapshots) ----
import numpy as np, matplotlib.pyplot as plt
from samheat.analysis.composites import open_3d, derived_fields
PREVIEW_RV, N_SNAP = RV_LIST[0], 8
K = MASK_PARAMS.get("k", 1)                     # W level shown

d = open_3d(EXPERIMENT / f"stage2_rv{PREVIEW_RV:g}", SOURCE, TMIN, TMAX)
z = d.z.values.astype(float)
idx = np.linspace(0, d.sizes["time"] - 1, N_SNAP).astype(int)
counts, capes, cins, bws = {}, [], [], []
for t in idx:
    s = d.isel(time=t).load()
    f = {v: s[v].values.astype(float) for v in d.data_vars if d[v].dims == ("time", "z", "y", "x")}
    f |= derived_fields(f, np.asarray(s["p"].values, float).reshape(-1, len(z))[0], z)
    m = MASKS[MASK](f, z, **MASK_PARAMS)
    for g, a in m.items():
        counts[g] = counts.get(g, 0) + a.sum()
    cape, cin = parcel_cape(f, z)
    B = buoyancy_at_w(buoyancy(f), K)
    capes.append(cape.ravel()); cins.append(cin.ravel()); bws.append(B.ravel())
total = len(idx) * f["W"][K].size
print(f"rv={PREVIEW_RV}, {N_SNAP} snapshots; mask {MASK}{MASK_PARAMS}")
for g, c in counts.items():
    print(f"  {g:16s} {100 * c / total:6.2f} %")

fig, ax = plt.subplots(2, 3, figsize=(17, 9))
x, y = d.x / 1e3, d.y / 1e3
panels = [(f["W"][K], "W at level %d (m/s)" % K, "RdBu_r", (-1, 1)), (1e2 * B, "buoyancy at that W level (cm/s$^2$)", "RdBu_r", (-3, 3)),
          (cape, "CAPE proxy (J/kg)", "viridis", None), (cin, "CIN proxy (J/kg)", "magma", None)]
for a, (fld, title, cmap, lim) in zip(ax.flat, panels):
    im = a.pcolormesh(x, y, fld, cmap=cmap, **(dict(vmin=lim[0], vmax=lim[1]) if lim else {}))
    fig.colorbar(im, ax=a); a.set_title(title); a.set_aspect("equal")
cat = np.full(f["W"][K].shape, np.nan)
for i, (g, a_) in enumerate(m.items()):
    cat[a_] = i
im = ax.flat[4].pcolormesh(x, y, cat, cmap=plt.get_cmap("tab10", len(m)), vmin=-.5, vmax=len(m) - .5)
cb = fig.colorbar(im, ax=ax.flat[4], ticks=range(len(m))); cb.ax.set_yticklabels(list(m)); ax.flat[4].set_aspect("equal")
ax.flat[4].set_title(f"groups, day {float(s.time):.2f}")
ax.flat[5].hist(np.concatenate(capes), bins=100, alpha=.6, label="CAPE")
ax.flat[5].hist(-np.concatenate(cins), bins=100, alpha=.6, label="-CIN")
ax.flat[5].set_yscale("log"); ax.flat[5].set_xlabel("J/kg"); ax.flat[5].legend(); ax.flat[5].set_title(f"{N_SNAP} snapshots")
plt.tight_layout(); plt.show()

## Compute composites for every r_v

In [ ]:
# ---- Composites for every r_v with the chosen thermodynamic mask (cached; the helpers are part of the key) ----
from samheat.analysis.composites import composite_experiment
C = composite_experiment(EXPERIMENT, RV_LIST, MASKS[MASK], MASK, MASK_PARAMS, source=SOURCE,
                         tmin=TMIN, tmax=TMAX, include_all=INCLUDE_ALL, key_extra=KEY_EXTRA)
C

In [ ]:
# ---- Group fractions (% of columns) vs r_v and near-surface RH ----
import pandas as pd
frac = (100 * C.fraction).to_pandas().round(2)
frac.insert(0, "RH (%)", C.RH.values.round(1))
frac

## Each group across r_v

In [ ]:
# ---- B. How each group changes with r_v: one figure per group, one line per r_v ----
# x-axes fit the data between 0 and ZMAX_PLOT, except the fixed ranges in XLIMS; dashed = saturated (MSE*, theta_e*)
# colours follow each run's near-surface RH (dry -> moist along CMAP)
import numpy as np
import matplotlib.pyplot as plt
from samheat.analysis.composites import SAT_PAIRS, fit_xlim
GROUPS_SHOW = [g for g in C.group.values if g != "all"]
QUANTS = ["W", "TABS", "QV", "QN", "QP", "MSE", "MSE_frozen", "THETA_E"]
ANOMALY = False           # True: plot group minus the domain mean ("all") of the same run
SHOW_SAT = True           # dashed saturated lines on the MSE / theta_e panels
XLIMS = {"MSE": (320, 350), "MSE_frozen": (320, 350), "THETA_E": (320, 350)}   # fixed axes (ignored if ANOMALY)
CMAP = plt.get_cmap("YlGnBu")

rh = C.RH.values
rh_norm = (rh - np.nanmin(rh)) / max(np.nanmax(rh) - np.nanmin(rh), 1e-9)
rv_color = {rv: CMAP(0.3 + 0.7 * n) for rv, n in zip(C.rv.values, rh_norm)}     # 0.3: skip the near-white end

for g in GROUPS_SHOW:
    ncol = 4
    nrow = -(-len(QUANTS) // ncol)
    fig, axs = plt.subplots(nrow, ncol, figsize=(4 * ncol, 4.2 * nrow), sharey=True, squeeze=False)
    for ax, q in zip(axs.flat, QUANTS):
        shown = []
        names = [q] + ([SAT_PAIRS[q]] if SHOW_SAT and q in SAT_PAIRS and SAT_PAIRS[q] in C else [])
        for rv in C.rv.values:
            for j, name in enumerate(names):
                prof = C[name].sel(rv=rv, group=g)
                if ANOMALY:
                    prof = prof - C[name].sel(rv=rv, group="all")
                ax.plot(prof, C.z / 1000, c=rv_color[rv], ls="--" if j else "-", lw=1 if j else 1.5,
                        label=None if j else f"rv {rv:g} (RH {float(C.RH.sel(rv=rv)):.0f} %, {100 * float(C.fraction.sel(rv=rv, group=g)):.1f} %)")
                shown.append(prof.values)
        if q in XLIMS and not ANOMALY:
            ax.set_xlim(*XLIMS[q])
        else:
            fit_xlim(ax, shown, C.z, ZMAX_PLOT)
        ax.set_title(q + (" - all" if ANOMALY else "") + (" (dashed: saturated)" if len(names) > 1 else ""))
        if ANOMALY:
            ax.axvline(0, c=".7", lw=.5)
    for ax in axs[:, 0]:
        ax.set_ylabel("z (km)")
    for ax in axs.flat[len(QUANTS):]:
        ax.set_visible(False)
    if ZMAX_PLOT:
        axs[0, 0].set_ylim(0, ZMAX_PLOT / 1000)
    axs.flat[0].legend(fontsize=7)
    fig.suptitle(f"{g}: mask {MASK}{MASK_PARAMS}")
    fig.tight_layout(); plt.show()

## Local stability of each group (Emanuel 6.1.7 / 6.2.10)

In [ ]:
# ---- D. Local stability (Emanuel 1994): N^2 (6.1.7) and N_m^2 (6.2.10) per group, one line per r_v ----
import numpy as np, matplotlib.pyplot as plt
from samheat.analysis.stability import stability_profiles
from samheat.analysis.composites import fit_xlim
CONDENSATE = ("QN",)          # condensate counted in r_T for N_m^2; ("QN", "QP") also counts falling rain
STAB_GROUPS = list(C.group.values)          # e.g. ["all", "updraft"]
PANELS = ["N2", "N2_m", "N2_m_entropy", "N2_m_loading"]

if not np.isfinite(C.p).all():
    print("WARNING: no mean pressure for some runs (STAT file missing?): stability will be NaN there")
ST = stability_profiles(C, CONDENSATE)

for g in STAB_GROUPS:
    fig, axs = plt.subplots(1, len(PANELS), figsize=(4 * len(PANELS), 4.5), sharey=True)
    for ax, q in zip(axs, PANELS):
        shown = []
        for i, rv in enumerate(ST.rv.values):
            prof = 1e4 * ST[q].sel(rv=rv, group=g)
            ax.plot(prof, ST.z / 1000, c=RV_COLORS[i % len(RV_COLORS)], label=f"rv {rv:g} (RH {float(C.RH.sel(rv=rv)):.0f} %)")
            shown.append(prof.values)
        fit_xlim(ax, shown + [np.zeros(ST.sizes["z"])], ST.z, ZMAX_PLOT)
        ax.axvline(0, c="k", lw=.6)
        ax.set_title(q); ax.set_xlabel("1e-4 s$^{-2}$")
    axs[0].set_ylabel("z (km)")
    if ZMAX_PLOT:
        axs[0].set_ylim(0, ZMAX_PLOT / 1000)
    axs[0].legend(fontsize=7)
    fig.suptitle(f"{g}: stability (Emanuel 6.1.7 / 6.2.10), r_T = r* + {'+'.join(CONDENSATE)}; mask {MASK}{MASK_PARAMS}")
    fig.tight_layout(); plt.show()